it extreact the required items first then will do the removal keyowrd search
- it first extract the description, androidmanifest path and readme then perform the search for removal keyword


In [ ]:
import os
import re
import requests
import pandas as pd
from base64 import b64decode
from dotenv import load_dotenv
from time import sleep

# === Load tokens ===
load_dotenv("All_Tokens.env")
tokens = [os.getenv(f"GITHUB_TOKEN_{i}") for i in range(1, 7)]
tokens = [t for t in tokens if t]
if not tokens:
    raise ValueError("❌ No GitHub tokens found in All_Tokens.env")
token_index = 0

def get_auth_header():
    global token_index
    token = tokens[token_index]
    token_index = (token_index + 1) % len(tokens)
    return {"Authorization": f"token {token}"}

# ✅ Output folder path
OUTPUT_DIR = r"C:\Android Mobile App\Step1_URL_Search\Type_1_Searching_Pipeline"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# === Define keyword baskets ===
keywords = ['example', 'sample', 'demo', 'test', 'debug', 'presentation', 'module', 'components', 'lib', 'library',
            'sdk', 'utils', 'utility', 'plugin', 'widget', 'playground', 'framework', 'architecture', 'skeleton',
            'collection', 'starting point', 'protocol', 'benchmark', 'hackathon', 'classroom', 'course', 'exercise',
            'assignment', 'homework', 'assessment', 'interview', 'asset', 'template', 'catalog', 'tutorial', 'tool']

preceded_by = ['this', 'is a', 'is an', 'our', 'my']
not_preceded_by = ['using', 'with']

# === GitHub API fetch functions ===
def fetch_repo_metadata(repo_full_name):
    url = f"https://api.github.com/repos/{repo_full_name}"
    resp = requests.get(url, headers=get_auth_header())
    if resp.status_code == 200:
        return resp.json().get("description", "")
    return ""

def fetch_manifest_paths(repo_full_name):
    url = f"https://api.github.com/search/code?q=filename:AndroidManifest.xml+repo:{repo_full_name}"
    resp = requests.get(url, headers=get_auth_header())
    if resp.status_code == 200:
        return [item["path"] for item in resp.json().get("items", [])]
    return []

def fetch_readme_content(repo_full_name):
    url = f"https://api.github.com/repos/{repo_full_name}/readme"
    resp = requests.get(url, headers=get_auth_header())
    if resp.status_code == 200:
        content = resp.json().get("content", "")
        return b64decode(content).decode('utf-8', errors='ignore')
    return ""

# === Load input ===
input_path = os.path.join(OUTPUT_DIR, "step2_manifest_final_output.csv")
output_path = os.path.join(OUTPUT_DIR, "step3_removal_keyword_output.csv")

if os.path.exists(output_path):
    df = pd.read_csv(output_path)
else:
    df = pd.read_csv(input_path)
    df = df[df["Valid_Repo_Step2"] == "yes"].copy()
    df["Repository"] = df["html_url"].apply(lambda url: '/'.join(url.strip('/').split('/')[-2:]))
    df["removal_keyword_flag"] = "none"
    df["removal_reason"] = ""
    df["Valid_Repo_Step3"] = ""

# === Detection with accurate indexing ===
to_review = df[df["Valid_Repo_Step3"].isna() | (df["Valid_Repo_Step3"] == "")]
for i, idx in enumerate(to_review.index, 1):
    row = df.loc[idx]
    repo = row['Repository']
    html_url = row['html_url']
    print(f"[{i}/{len(to_review)}] Reviewing {repo}...")

    removal_sources = []

    try:
        description = fetch_repo_metadata(repo)
        manifest_paths = fetch_manifest_paths(repo)
        readme = fetch_readme_content(repo)

        if any(k in path.lower() for k in keywords for path in manifest_paths):
            removal_sources.append("manifest_path")

        if any(k in repo.lower() for k in keywords):
            removal_sources.append("repo_name")

        if any(
            re.search(rf'(?<!\S){re.escape(k)}(?!\S)', str(description), re.IGNORECASE) and
            not any(re.search(rf'(?<!\S){re.escape(word)}\s+(\S+\s+){{0,4}}{re.escape(k)}(?!\S)', str(description), re.IGNORECASE)
                    for word in not_preceded_by)
            for k in keywords
        ):
            removal_sources.append("description")

        if any(
            re.search(rf'(?<!\S){re.escape(k)}(?!\S)', readme, re.IGNORECASE) and
            any(re.search(rf'(?<!\S){re.escape(p)}\s+(\S+\s+){{0,4}}{re.escape(k)}(?!\S)', readme, re.IGNORECASE)
                for p in preceded_by)
            for k in keywords
        ):
            removal_sources.append("readme")

        flag = "yes" if removal_sources else "no"
        df.at[idx, "removal_keyword_flag"] = flag
        df.at[idx, "Valid_Repo_Step3"] = "no" if flag == "yes" else "yes"
        df.at[idx, "removal_reason"] = ", ".join(removal_sources)

    except Exception as e:
        print(f"❌ Error processing {repo}: {e}")
        df.at[idx, "removal_keyword_flag"] = "error"
        df.at[idx, "Valid_Repo_Step3"] = "no"
        df.at[idx, "removal_reason"] = "error"
        sleep(1)

    df.to_csv(output_path, index=False)

print(f"✅ Step 3 complete: {output_path} saved.")


[16/28250] Reviewing sintaxi/phonegap...
[17/28250] Reviewing rhomobile/rhodes...
[23/28250] Reviewing bradfitz/android-garage-opener...
[26/28250] Reviewing Dawnthorn/nagare...
[28/28250] Reviewing jamplus/jamplus...
[33/28250] Reviewing bpellin/keepassdroid...
[45/28250] Reviewing samuelclay/NewsBlur...
[48/28250] Reviewing connectbot/connectbot...
[61/28250] Reviewing JakeWharton/SMSMorse...
[62/28250] Reviewing JakeWharton/SMSBarrage...
[68/28250] Reviewing millenomi/diceshaker...
[84/28250] Reviewing simpligility/android-maven-plugin...
[99/28250] Reviewing pocmo/Yaaic...
[102/28250] Reviewing ushahidi/Ushahidi_Android...
[120/28250] Reviewing Ramblurr/Anki-Android...
[132/28250] Reviewing novoda/android-demos...
[133/28250] Reviewing commonsguy/cw-advandroid...
[135/28250] Reviewing commonsguy/cwac-merge...
[137/28250] Reviewing konklone/congress-android...
[140/28250] Reviewing johannilsson/sthlmtraveling...
[145/28250] Reviewing tidev/titanium-sdk...
[148/28250] Reviewing appce

KeyboardInterrupt: 